# WMDP Target-Model Mini GPU Test

This notebook runs a real target model on WMDP bio/chem/cyber with `sample_size=2`, no corruption, and GPU required. It is the gate before running the target model on the full WMDP dataset.

Defaults: `PORT_TARGET_CONFIG_NAME=phi-1_5`, `PORT_ATTN_IMPLEMENTATION=eager`, `PORT_TORCH_DTYPE=float16`, `PORT_WMDP_BATCH_SIZE=1`.


In [ ]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys
import time

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()

def has_project_layout(path):
    path = Path(path)
    return (path / 'llm-unlearn-eco' / 'eco').exists() and (path / 'dataset').exists()

def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()

PROJECT_ROOT = clone_or_use_project()
ECO_ROOT = PROJECT_ROOT / 'llm-unlearn-eco'
os.environ['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
if str(ECO_ROOT) not in sys.path:
    sys.path.insert(0, str(ECO_ROOT))

commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Project root: {PROJECT_ROOT}')
print(f'Commit: {commit_sha}')


In [ ]:
required_packages = {
    'datasets': 'datasets>=2.10.1',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow>=10',
    'tabulate': 'tabulate',
    'tqdm': 'tqdm',
    'transformers': 'transformers>=4.38.0',
    'accelerate': 'accelerate>=0.26.0',
    'yaml': 'pyyaml',
}

missing_packages = []
for module_name, package_spec in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_packages.append(package_spec)

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('Required packages are already available.')


In [ ]:
import gc
import platform
import pandas as pd
import torch
import yaml

from eco.dataset import WMDPBio, WMDPChem, WMDPCyber
from eco.evaluator import ChoiceByTopLogit
from eco.inference import EvaluationEngine
from eco.model import HFModel
from eco.paths import MODEL_CONFIG_DIR

print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is required for this target-model mini test. Enable a Kaggle GPU runtime and rerun.')

for idx in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(idx)
    print(f'GPU {idx}: {props.name}, VRAM={props.total_memory / 1024**3:.2f} GiB')


## Runtime Model Config

The notebook copies a repo model config to the Kaggle working directory and overrides attention/dtype for a conservative GPU mini test. This avoids editing checked-in full-run configs just to smoke test a target model.


In [ ]:
TARGET_CONFIG_NAME = os.environ.get('PORT_TARGET_CONFIG_NAME', 'phi-1_5')
TARGET_HF_NAME = os.environ.get('PORT_TARGET_HF_NAME')
TARGET_MODEL_PATH = os.environ.get('PORT_TARGET_MODEL_PATH') or None
ATTN_IMPLEMENTATION = os.environ.get('PORT_ATTN_IMPLEMENTATION', 'eager')
TORCH_DTYPE = os.environ.get('PORT_TORCH_DTYPE', 'float16')
BATCH_SIZE = int(os.environ.get('PORT_WMDP_BATCH_SIZE', '1'))
SAMPLE_SIZE = 2

base_config_path = MODEL_CONFIG_DIR / f'{TARGET_CONFIG_NAME}.yaml'
if not base_config_path.exists():
    raise FileNotFoundError(f'Model config not found: {base_config_path}')

with open(base_config_path, 'r', encoding='utf-8') as f:
    runtime_config = yaml.safe_load(f)

if TARGET_HF_NAME:
    runtime_config['hf_name'] = TARGET_HF_NAME
runtime_config['attn_implementation'] = ATTN_IMPLEMENTATION
runtime_config['torch_dtype'] = TORCH_DTYPE
runtime_config['load_in_8bit'] = bool(runtime_config.get('load_in_8bit', False))
runtime_config['load_in_4bit'] = bool(runtime_config.get('load_in_4bit', False))

RUN_NAME = f'wmdp_target_mini_no_corrupt_{TARGET_CONFIG_NAME}'
RUN_DIR = (Path('/kaggle/working') if IS_KAGGLE else PROJECT_ROOT / 'results') / RUN_NAME
RUNTIME_CONFIG_DIR = RUN_DIR / 'model_config'
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_MODEL_NAME = f'{TARGET_CONFIG_NAME}_runtime_gpu_mini'
runtime_config_path = RUNTIME_CONFIG_DIR / f'{RUNTIME_MODEL_NAME}.yaml'
with open(runtime_config_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(runtime_config, f, sort_keys=False)

run_config = {
    'repo_url': REPO_URL,
    'commit_sha': commit_sha,
    'target_config_name': TARGET_CONFIG_NAME,
    'target_hf_name': runtime_config.get('hf_name'),
    'target_model_path': TARGET_MODEL_PATH,
    'runtime_model_name': RUNTIME_MODEL_NAME,
    'runtime_config_path': str(runtime_config_path),
    'attn_implementation': ATTN_IMPLEMENTATION,
    'torch_dtype': TORCH_DTYPE,
    'batch_size': BATCH_SIZE,
    'sample_size': SAMPLE_SIZE,
    'corrupt_method': None,
    'datasets': ['wmdp-bio', 'wmdp-chem', 'wmdp-cyber'],
    'run_dir': str(RUN_DIR),
}
(RUN_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')
print(json.dumps(run_config, indent=2))


## Load Target Model


In [ ]:
start_model = time.perf_counter()
model = HFModel(
    model_name=RUNTIME_MODEL_NAME,
    model_path=TARGET_MODEL_PATH,
    config_path=str(RUNTIME_CONFIG_DIR),
)
if model.tokenizer.pad_token_id is None:
    model.tokenizer.pad_token = model.tokenizer.eos_token
model.tokenizer.padding_side = 'right'
model_load_seconds = time.perf_counter() - start_model
print(f'Model loaded in {model_load_seconds:.2f}s')
if torch.cuda.is_available():
    print('CUDA memory allocated GiB:', torch.cuda.memory_allocated() / 1024**3)
    print('CUDA memory reserved GiB:', torch.cuda.memory_reserved() / 1024**3)


## Run WMDP Sample Size 2, No-Corrupt


In [ ]:
choice_labels = ['A', 'B', 'C', 'D']
all_summaries = []
all_predictions = []
subject_timings = {}

for data_module in [WMDPBio(sample_size=SAMPLE_SIZE), WMDPChem(sample_size=SAMPLE_SIZE), WMDPCyber(sample_size=SAMPLE_SIZE)]:
    print(f'\nRunning {data_module.name} target mini test...')
    subject_start = time.perf_counter()
    engine = EvaluationEngine(
        model=model,
        tokenizer=model.tokenizer,
        data_module=data_module,
        subset_names=['test'],
        evaluator=ChoiceByTopLogit(),
        batch_size=BATCH_SIZE,
    )
    engine.inference()
    summary_stats, outputs = engine.summary()
    elapsed = time.perf_counter() - subject_start
    subject_timings[data_module.name] = elapsed
    all_summaries.extend(summary_stats)

    result_name, batches = list(outputs[0].items())[0]
    row_idx = 0
    for batch in batches:
        for c, p in zip(batch['correct'], batch['predicted']):
            all_predictions.append({
                'dataset': data_module.name,
                'row_index': row_idx,
                'correct_index': int(c),
                'predicted_index': int(p),
                'correct_label': choice_labels[int(c)] if 0 <= int(c) < len(choice_labels) else str(c),
                'predicted_label': choice_labels[int(p)] if 0 <= int(p) < len(choice_labels) else str(p),
                'is_correct': int(c) == int(p),
            })
            row_idx += 1
    print(f'{data_module.name} completed in {elapsed:.2f}s with {row_idx} rows.')

predictions_df = pd.DataFrame(all_predictions)
summary_by_dataset = predictions_df.groupby('dataset')['is_correct'].agg(['count', 'mean']).reset_index()
summary_by_dataset = summary_by_dataset.rename(columns={'mean': 'accuracy'})
print(summary_by_dataset)


In [ ]:
summary_payload = {
    'run_config': run_config,
    'model_load_seconds': model_load_seconds,
    'subject_timings_seconds': subject_timings,
    'engine_summaries': all_summaries,
    'summary_by_dataset': summary_by_dataset.to_dict(orient='records'),
    'total_rows': int(len(predictions_df)),
    'overall_accuracy': float(predictions_df['is_correct'].mean()) if len(predictions_df) else None,
}

summary_path = RUN_DIR / 'summary.json'
predictions_path = RUN_DIR / 'predictions.csv'
summary_by_dataset_path = RUN_DIR / 'summary_by_dataset.csv'
summary_path.write_text(json.dumps(summary_payload, indent=2, default=str), encoding='utf-8')
predictions_df.to_csv(predictions_path, index=False)
summary_by_dataset.to_csv(summary_by_dataset_path, index=False)

print(f'Wrote {summary_path}')
print(f'Wrote {predictions_path}')
print(f'Wrote {summary_by_dataset_path}')
print(json.dumps(summary_payload, indent=2, default=str)[:4000])


In [ ]:
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('WMDP TARGET-MODEL MINI GPU TEST COMPLETED')
